In [1]:
import polars as pl
from huggingface_hub import hf_hub_download 
from huggingface_hub import snapshot_download

local_path_train = hf_hub_download(
    repo_id="cais/mmlu",
    filename="all/auxiliary_train-00000-of-00001.parquet",
    repo_type="dataset"
)
df = pl.read_parquet(local_path_train)
df

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

answer,choices,question,subject
i64,list[str],str,str
1,"[""Adams only."", ""Brooks only."", … ""Adams and Brooks""]","""Davis decided to kill Adams. H…",""""""
3,"[""guilty, because this is a public welfare offense. "", ""guilty, because he cannot be excused on the basis of his own failure to keep proper records. "", … ""not guilty, because he was not aware of the value of the drugs he had sold""]","""A state statute requires any p…",""""""
2,"[""Yes, because Mann threatened to use deadly force. "", ""Yes, unless Mann was related to Borrower. "", … ""No, because Lender was the original aggressor by threatening Borrower with a battery""]","""Lender met Borrower on the str…",""""""
1,"[""must permit Don to answer if he had objected to Peter's testimony."", ""may permit Don to answer, whether or not he had objected to Peter's testimony. "", … ""cannot permit Don to answer, whether or not he had objected to Peter's testimony""]","""Peter sued Don for breach of c…",""""""
2,"[""partial breach of contract only if Ames had properly or substantially painted the porch."", ""partial breach of contract whether or not Ames had properly or substantially painted the porch."", … ""total breach of contract whether or not Ames had properly or substantially painted the porch""]","""Ames had painted Bell's house …",""""""
…,…,…,…
0,"[""skyscrapers"", ""the stock market"", … ""water""]","""A bulldozer alters the area of""",""""""
0,"[""Brewer's yeast"", ""air"", … ""sugar""]","""An organism that can survive w…",""""""
1,"[""the praying mantis"", ""the antelope"", … ""the sloth""]","""The nimbleness of this animal …",""""""


In [ ]:
from tinyreasoner import chat_api

def load_w_choices():
    local_path_train = hf_hub_download(
        repo_id="cais/mmlu",
        filename="all/auxiliary_train-00000-of-00001.parquet",
        repo_type="dataset"
    )
    df = pl.read_parquet(local_path_train)
    samples = []

    for row in df.iter_rows(named=True):

        choices = row["choices"]
        question = row["question"]
        answer = choices[row["answer"]]

        choices_str = ' - ' + '\n - '.join(choices)
        items = [
            chat_api.UserMessage(f"{question}. Write out the correct choice:\n{choices_str}"),
            chat_api.AssistantMessage(f"{answer}")
        ]

        samples.append(items)
    return samples

def load_w_numbered_choices():
    local_path_train = hf_hub_download(
        repo_id="cais/mmlu",
        filename="all/auxiliary_train-00000-of-00001.parquet",
        repo_type="dataset"
    )
    df = pl.read_parquet(local_path_train)
    samples = []

    for row in df.iter_rows(named=True):

        choices = row["choices"]
        question = row["question"]
        answer = row["answer"]

        choices_str = ""
        for i,c in enumerate(choices, start=1):
            choices_str = f'{choices_str}\n{i}: {c}'

        items = [
            chat_api.UserMessage(f"{question}. Write the number of the correct choice :{choices_str}"),
            chat_api.AssistantMessage(f"{answer + 1}")
        ]

        samples.append(items)
    return samples


def load_w_o_choices():
    local_path_train = hf_hub_download(
        repo_id="cais/mmlu",
        filename="all/auxiliary_train-00000-of-00001.parquet",
        repo_type="dataset"
    )
    df = pl.read_parquet(local_path_train)
    samples = []

    for row in df.iter_rows(named=True):

        choices = row["choices"]
        question = row["question"]
        answer = choices[row["answer"]]

        items = [
            chat_api.UserMessage(question),
            chat_api.AssistantMessage(answer)
        ]

        samples.append(items)
    return samples

def load() -> list[list[chat_api.Item]]:
    return load_w_choices() + load_w_numbered_choices() + load_w_o_choices()

In [1]:
from tinyreasoner.datasets.synthetic_puzzles import load

load()[0]

[UserMessage(role=user, content=Given dependencies: A->B, B->C, C->D, D->E, E->F, A->F, is there a valid topological ordering of 6 tasks?, type=user),
 AssistantStart(type=assistant_start),
 AssistantMessage(role=assistant, content=yes, type=assistant),
 AssistantEnd(type=assistant_end)]